## Reading 36-1 - Getting Time Right

By the end of this reading, you will learn:
<ol>
    <li>Six rules for handling time in embedded and flight software</li>
    <li>How long a counter can run before it wraps around, and two real systems that failed when theirs did</li>
    <li>How to write a timeout that still works when the counter wraps</li>
    <li>Why storing time as integers is necessary but not sufficient</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading36
    cd reading36

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec36/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    counter_limits.c  Makefile  tick_drift.c  torn_read.c  wraparound.c

### A Correct Value at the Wrong Time Is a Wrong Value

In Lecture 33, the Patriot battery's clock lost 0.34 seconds over 100 hours, and that was enough to miss an incoming missile. Time is an input to almost every decision flight software makes: when to fire a thruster, whether a sensor reading is still fresh, when a command has waited too long. If the time is wrong, every decision that uses it is wrong too.

Amalaye Oyake summarizes how flight software teams handle time in six rules:

<table>
    <tr><th>Rule</th><th>What it prevents</th></tr>
    <tr><td><b>1.</b> Keep one continuous time scale onboard, such as TAI or GPS time. Convert to UTC only at the interfaces.</td><td>UTC and Unix time jump when a leap second is added. Today, TAI is 37 seconds ahead of UTC, and GPS time is 18 seconds ahead.</td></tr>
    <tr><td><b>2.</b> Store time as integers: whole seconds plus a binary fraction.</td><td>Rounding errors that grow, as in the Patriot failure.</td></tr>
    <tr><td><b>3.</b> Time-stamp data at the source, in hardware.</td><td>The time a value <i>arrives</i> includes operating-system delays that change from run to run.</td></tr>
    <tr><td><b>4.</b> Carry the age of every value, and reject stale data.</td><td>Acting on an attitude estimate from before the vehicle moved.</td></tr>
    <tr><td><b>5.</b> Understand that your oscillator will drift.</td><td>A clock crystal's frequency changes with temperature and age, so onboard time slowly diverges from true time.</td></tr>
    <tr><td><b>6.</b> Prove deadlines with worst-case timing analysis, and monitor for missed deadlines at run time.</td><td>Average-case tests hide worst-case paths. Mars Pathfinder missed deadlines through priority inversion.</td></tr>
</table>

This reading focuses on Rules 2 and 4, which come down to how you write C. Reading 36-2 covers a problem behind Rule 6: what happens when two parts of a program share a clock.

### Every Counter Has a Lifetime - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec36/counter_limits.c">counter_limits.c</a>

Flight software keeps time by counting ticks in an integer. In Lecture 32, we saw that an integer has a fixed number of bits, so every counter eventually runs out. How long does that take?

    $ make counter_limits
    uint32_t milliseconds                          49.7 days
    int32_t hundredths of a second                248.6 days
    uint32_t seconds                                136 years
    uint64_t nanoseconds                            585 years

Two of those numbers have caused real failures:
<ul>
    <li><b>49.7 days:</b> Windows 95 and 98 kept a 32-bit count of milliseconds since startup. Microsoft confirmed that a computer left running for 49.7 days could hang when that counter wrapped back to zero. It took years for anyone to notice, because very few people left a Windows 95 computer on for seven weeks.</li>
    <li><b>248 days:</b> In 2015, the FAA issued an emergency directive for the Boeing 787. A counter in the generator control units would overflow after 248 days of continuous power, putting all four units into a fail-safe mode at the same time and cutting off the aircraft's AC electrical power, possibly in flight. Until a software fix was installed, airlines were required to power-cycle the aircraft before it reached 248 days.</li>
</ul>

248.6 days is exactly the lifetime of a <b>signed</b> 32-bit count of hundredths of a second. From Lecture 32, you know that signed overflow is undefined behavior, so the program's behavior after day 248 is not even defined by the C language.

Spacecraft run for years, and some, like Voyager, for decades. "Long enough" has to be calculated, not assumed.

### Timeouts That Survive the Wraparound - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec36/wraparound.c">wraparound.c</a>

Sometimes a counter <i>must</i> wrap, because a 32-bit timer is all the hardware provides. The code that uses it must still work. Consider a command timeout of 1000 ms, where the command was sent 300 ms before the counter wraps to zero. Here are two ways to check it:

    /* WRONG: the sum start + TIMEOUT_MS can wrap around to a small number */
    int timed_out_naive(uint32_t now, uint32_t start)
    {
    	return now >= start + TIMEOUT_MS;
    }

    /* RIGHT: unsigned subtraction wraps too, so it always gives elapsed time */
    int timed_out_safe(uint32_t now, uint32_t start)
    {
    	return now - start >= TIMEOUT_MS;
    }

    $ make wraparound
    start = 4294966996   start + TIMEOUT_MS = 700

             now   elapsed ms      naive       safe
      4294966996            0    TIMEOUT    waiting
      4294967096          100    TIMEOUT    waiting
      4294967196          200    TIMEOUT    waiting
               0          300    waiting    waiting
             ...
             700         1000    TIMEOUT    TIMEOUT

The naive version reports a timeout <b>immediately</b>, at 0 ms. <code>start + TIMEOUT_MS</code> wrapped around to 700, and every value of <code>now</code> before the wrap is larger than 700.

The safe version works because <b>unsigned arithmetic in C is defined to wrap</b>. When <code>now</code> is 0 and <code>start</code> is 4294966996, <code>now - start</code> wraps to exactly 300, the true elapsed time. As long as the interval you measure is shorter than the counter's lifetime, subtraction always gives the right answer.

<b>The rule:</b> never compare two raw timestamps, and never add to a timestamp before comparing. Subtract to get the elapsed time, then compare that.

The same subtraction implements Ama's Rule 4. Every value carries the time it was measured, and before using it:

    uint32_t age_ms = now - value.timestamp;
    if (age_ms > MAX_AGE_MS) {
    	/* stale: do not use it */
    }

### Integers Are Not Enough - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec36/tick_drift.c">tick_drift.c</a>

The CCSDS standards that most space agencies use store time as whole seconds plus a <b>binary fraction</b>, such as units of 1/65536 of a second. Why binary? Many clock crystals oscillate at a power of two. The 32,768 Hz crystal in most watches and real-time clocks is 2<sup>15</sup> Hz, so its ticks fit a binary fraction exactly.

But suppose the timer ticks every 0.1 s. In units of 1/65536 s, 0.1 s is 6553.6, which is not an integer. Here are three ways to keep this clock for one day:

    $ make tick_drift
    Expected after one day:        86400.000000 s
    1. float accumulation:         87145.843750 s
    2. integer, rounded increment: 86405.273438 s
    3. integer tick count:         86400.000000 s

Method 1 is the float clock from Reading 33-1. Method 2 uses integers, but adds the <i>rounded</i> value 6554 every tick. Each tick adds 0.4/65536 of a second too much, and over 864,000 ticks that is 5.27 seconds per day. This is the same mistake as the Patriot, made with integers.

Method 3 counts ticks, which is exact, and converts the total only when it needs to:

    exact.seconds = ticks / 10U;
    exact.fraction = (ticks % 10U) * FRACTION_PER_SECOND / 10U;

<b>The lesson:</b> integers alone do not make time exact. Count in the hardware's own unit, and never accumulate a rounded value.

### <font color = "red">Class Introduction Question #1 - Why does a signed 32-bit count of hundredths of a second fail after 248.6 days, and why is the program's behavior after that point not even defined by C?</font>

### <font color = "red">Class Introduction Question #2 - Explain why <code>now - start >= TIMEOUT_MS</code> gives the right answer across a wraparound, while <code>now >= start + TIMEOUT_MS</code> does not.</font>

### <font color = "red">Class Introduction Question #3 - Method 2 in <code>tick_drift.c</code> uses only integers, but still drifts by more than 5 seconds per day. Why, and how does Method 3 avoid it?</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec36/Reading%2036-2.ipynb">Reading 36-2 - Shared Clocks and Torn Reads</a>